# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method selected: Random Forest Classifier.** The task is to rank pages by the probability that their later impressions decline, so a classifier's probability score is more useful than a hard class label. A Random Forest captures non-linear relationships and interactions among impressions, position, CTR, activity, and content depth without requiring manual transformations, while its shallow trees and minimum leaf size limit overfitting. I use fixed seeds for reproducibility and keep the feature set restricted to information available by March 15.

In [5]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.inspection import permutation_importance

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Features stop on March 15; the label is defined only from the later outcome window.
dataset = con.sql(f"""
    WITH features AS (
        SELECT
            f.content_hash_id AS content_id,
            f.client_hash_id AS client_id,
            c.word_count,
            SUM(f.gsc_impressions) AS feat_impressions_15d,
            SUM(f.gsc_clicks) AS feat_clicks_15d,
            AVG(NULLIF(f.gsc_avg_position, 0)) AS feat_avg_position_15d,
            CASE WHEN SUM(f.gsc_impressions) > 0
                 THEN SUM(f.gsc_clicks)::FLOAT / SUM(f.gsc_impressions)
                 ELSE 0 END AS feat_ctr_15d,
            COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS feat_active_days_15d
        FROM read_parquet('{month_path}') f
        LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
          ON f.content_hash_id = c.content_hash_id
        WHERE f.report_date <= '2026-03-15'
        GROUP BY f.content_hash_id, f.client_hash_id, c.word_count
    ),
    outcomes AS (
        SELECT
            content_hash_id AS content_id,
            CASE WHEN SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END)
                      < SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END)
                 THEN 1 ELSE 0 END AS is_declining_label
        FROM read_parquet('{month_path}')
        GROUP BY content_hash_id
    )
    SELECT f.*, o.is_declining_label
    FROM features f
    INNER JOIN outcomes o ON f.content_id = o.content_id
""").df().fillna(0)

feature_cols = [
    'feat_impressions_15d',
    'feat_clicks_15d',
    'feat_avg_position_15d',
    'feat_ctr_15d',
    'feat_active_days_15d',
    'word_count'
]

print(f"Loaded {len(dataset):,} rows across {dataset['client_id'].nunique():,} clients")
print(f"Declining label rate: {dataset['is_declining_label'].mean():.3f}")

Loaded 319,759 rows across 52 clients
Declining label rate: 0.208


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(dataset, groups=dataset['client_id']))

train_df = dataset.iloc[train_idx].copy()
test_df = dataset.iloc[test_idx].copy()

X_train = train_df[feature_cols]
y_train = train_df['is_declining_label']
X_test = test_df[feature_cols]
y_test = test_df['is_declining_label']

assert set(train_df['client_id']).isdisjoint(set(test_df['client_id']))
print(f"Train set: {len(train_df):,} rows across {train_df['client_id'].nunique():,} clients")
print(f"Test set:  {len(test_df):,} rows across {test_df['client_id'].nunique():,} clients")
print(f"Train decline rate: {y_train.mean():.3f}; test decline rate: {y_test.mean():.3f}")

Train set: 260,708 rows across 39 clients
Test set:  59,051 rows across 13 clients
Train decline rate: 0.217; test decline rate: 0.168


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

test_df['ml_score'] = model.predict_proba(X_test)[:, 1]

# Reproduce the Week-4 rule on this exact held-out test split.
max_impressions = max(float(test_df['feat_impressions_15d'].max()), 1.0)
norm_impressions = np.log1p(test_df['feat_impressions_15d']) / np.log1p(max_impressions)
low_ctr_penalty = 1.0 - (
    test_df['feat_clicks_15d'] / (test_df['feat_impressions_15d'] + 1.0)
)
thin_content_flag = (test_df['word_count'] < 1000).astype(float) * 0.2
test_df['baseline_score'] = (
    0.5 * norm_impressions + 0.3 * low_ctr_penalty + 0.2 * thin_content_flag
)

k = min(50, len(test_df))
precision_at_50_baseline = test_df.nlargest(k, 'baseline_score')['is_declining_label'].mean()
precision_at_50_ml = test_df.nlargest(k, 'ml_score')['is_declining_label'].mean()

comparison_table = pd.DataFrame([
    {
        'System': 'Week-4 Heuristic Baseline',
        'Precision@50': precision_at_50_baseline,
        'ROC-AUC': roc_auc_score(y_test, test_df['baseline_score']),
        'Brier Score': brier_score_loss(y_test, test_df['baseline_score'])
    },
    {
        'System': 'Week-5 Random Forest Model',
        'Precision@50': precision_at_50_ml,
        'ROC-AUC': roc_auc_score(y_test, test_df['ml_score']),
        'Brier Score': brier_score_loss(y_test, test_df['ml_score'])
    }
]).round(3)

print(f"Test base rate: {y_test.mean():.3f}")
display(comparison_table)
print('Note: the heuristic score is used for ranking; its Brier score is a diagnostic, not a calibration claim.')

Test base rate: 0.168


,System,Precision@50,ROC-AUC,Brier Score
0,Week-4 Heuristic Baseline,0.46,0.767,0.183
1,Week-5 Random Forest Model,0.74,0.856,0.117


Note: the heuristic score is used for ranking; its Brier score is a diagnostic, not a calibration claim.


## 4. Errors and interpretation

The table below reports model-driven feature importance and examples of errors at a 0.5 probability threshold. Feature importance is descriptive rather than causal: it tells us what the fitted model used, not what an editor can change with certainty. False positives can be pages with weak recent activity or thin content that nevertheless held traffic because of durable demand, brand intent, or strong domain authority. False negatives can be pages with healthy historical signals that then experienced an abrupt ranking change, SERP change, or other event not represented in the pre-cutoff features. These cases are useful review candidates because the model sees only the March 1–15 snapshot and cannot observe those unmeasured causes.

In [8]:
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

predicted_declining = test_df['ml_score'] >= 0.5
error_df = test_df[['content_id', 'ml_score', 'baseline_score', 'is_declining_label']].copy()
error_df['error_type'] = np.select(
    [predicted_declining & (error_df['is_declining_label'] == 0),
     ~predicted_declining & (error_df['is_declining_label'] == 1)],
    ['false_positive', 'false_negative'],
    default='correct'
)

print('Random Forest feature importance')
display(importance_df)
print('\nThree highest-confidence errors by model score distance from 0.5')
error_examples = error_df[error_df['error_type'] != 'correct'].copy()
error_examples['distance_from_threshold'] = (error_examples['ml_score'] - 0.5).abs()
display(
    error_examples.nlargest(3, 'distance_from_threshold')
    [['content_id', 'ml_score', 'baseline_score', 'is_declining_label', 'error_type']]
)
print(f"False positives: {(error_df['error_type'] == 'false_positive').sum():,}")
print(f"False negatives: {(error_df['error_type'] == 'false_negative').sum():,}")

Random Forest feature importance


,Feature,Importance
0,feat_impressions_15d,0.341811
1,feat_active_days_15d,0.341602
2,feat_avg_position_15d,0.219936
3,feat_ctr_15d,0.043654
4,word_count,0.031548
5,feat_clicks_15d,0.021449



Three highest-confidence errors by model score distance from 0.5


,content_id,ml_score,baseline_score,is_declining_label,error_type
142933,content_8cdfdef156238947,0.075967,0.682800,1,false_negative
317276,content_5f2e3544f3056924,0.076946,0.625869,1,false_negative
72395,content_a11bd5663919f057,0.870783,0.730902,0,false_positive


False positives: 8,129
False negatives: 3,832


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The model and Week-4 baseline use the same grouped test split and report Precision@50 and ROC-AUC
- [x] Feature importance and false-positive/false-negative examples are included
- [x] Committed to my repo under `work/notebooks/` — run the commit separately after review